# Tidy Data

¿Por qué necesitamos tidy data?
* En general los datos vienen en diferentes formatos 
* **Todos** los proyectos de ciencia de datos van a requerir que limpies los datos 
* Data tidying consiste en estructurar las bases para facilitar el analisis 

![](imagenes/tidydata_1.jpeg)


# ¿Qué hace que una base sea tidy?
![alternative text](imagenes/tidydata_2.jpeg)


# Los tidy data hacen que la ciencia de datos sea más eficiente
![alternative text](imagenes/tidydata_3.jpeg)

# Nuestro objetivo: transformar y wrangler los datos para que sea tidy 
![alternative text](imagenes/tidydata_4.jpeg)


# ¿Cómo hacemos que nuestros datos sean tidy?

Identificar que tipo de problema tenemos 
1. Datos dentro del nombre de las columnas 
2. Una observación en muchos renglones 
3. Una celda tiene dos datos 
4. Dos celdas tienen un solo dato 
5. Los datos están divididos a lo largo de varias columnas 
6. Los datos están divididos a lo largo de distintas tablas

## 1. Datos dentro del nombre de las columnas 

In [1]:
import pandas as pd 
base_url = "https://github.com/byuidatascience/data4python4ds/raw/master/data-raw/"
table4a = pd.read_csv("{}table4a/table4a.csv".format(base_url))

In [2]:
table4a

,country,1999,2000
0,Afghanistan,745,2666
1,Brazil,37737,80488
2,China,212258,213766


### ¿Qué está mal acá?
![alternative text](imagenes/tidytable1a.png)

### ¿Cómo lo arreglamos?
![alternative text](imagenes/tidy1b.png)

### Datos en las columnas $\implies$ `melt`

Vamos a crear un ejemplo desde cero antes de arreglar el que tenemos.

In [3]:
players = pd.DataFrame({'Game':['Athletic', 'Valencia'],
'Aubameyang':[1,3], 'Dembélé':[1,0], 'de Jong':[1,1], 
'Depay': [1,0]})
players

,Game,Aubameyang,Dembélé,de Jong,Depay
0,Athletic,1,1,1,1
1,Valencia,3,0,1,0


In [7]:
# Necesitamos que las columnas se vuelva una sola columna 
pd.melt(players, id_vars = 'Game', var_name = 'Jugador',value_name = 'Goles')

,Game,Jugador,Goles
0,Athletic,Aubameyang,1
1,Valencia,Aubameyang,3
2,Athletic,Dembélé,1
3,Valencia,Dembélé,0
4,Athletic,de Jong,1
5,Valencia,de Jong,1
6,Athletic,Depay,1
7,Valencia,Depay,0


## 2. Datos dentro del nombre de las columnas 

In [8]:
table2 = pd.read_csv("{}table2/table2.csv".format(base_url))
table2

,country,year,type,count
0,Afghanistan,1999,cases,745
1,Afghanistan,1999,population,19987071
2,Afghanistan,2000,cases,2666
3,Afghanistan,2000,population,20595360
4,Brazil,1999,cases,37737
5,Brazil,1999,population,172006362
6,Brazil,2000,cases,80488
7,Brazil,2000,population,174504898
8,China,1999,cases,212258
9,China,1999,population,1272915272


### ¿Qué está mal acá?
![alternative text](imagenes/tidytable2a.png)

### ¿Cómo lo arreglamos?
![alternative text](imagenes/tidytable2b.png)

### Una observación en varios renglones $\implies$ `pivot_table`

In [9]:
pd.melt(players, id_vars = 'Game', 
        var_name = 'Jugador', 
        value_name = 'Goles').pivot_table(index = 'Game',
                                         columns = 'Jugador',
                                         values = 'Goles').reset_index()

Jugador,Game,Aubameyang,Dembélé,Depay,de Jong
0,Athletic,1,1,1,1
1,Valencia,3,0,0,1


### EJERCICIO EN CLASE 
* Utiliza `melt` y `pivot_table` para arreglar las tablas: table2 y table4
* ¿Qué columnas elegiste?
* ¿Encontraste algún problema qué no existia en el ejemplo que mostramos antes?

## 3. Una celda tiene dos datos 

In [10]:
table3 = pd.read_csv("{}table3/table3.csv".format(base_url))
table3

,country,year,rate
0,Afghanistan,1999,745/19987071
1,Afghanistan,2000,2666/20595360
2,Brazil,1999,37737/172006362
3,Brazil,2000,80488/174504898
4,China,1999,212258/1272915272
5,China,2000,213766/1280428583


### ¿Qué está mal acá?
![alternative text](imagenes/table3a.png)

### ¿Cómo lo arreglamos?
![alternative text](imagenes/table3b.png)

### Una celda tiene dos datos $\implies$ `str.split  + pd.concat`

In [13]:
resultados  = pd.DataFrame({'Player': 
  ['Lewandowski', 'Dembélé', 'Fati', 'Pedri'], 
  'results':['12 4', '3 5' , '3 3', '2 0']})
resultados

,Player,results
0,Lewandowski,12 4
1,Dembélé,3 5
2,Fati,3 3
3,Pedri,2 0


In [14]:
nuevas_columnas = (resultados.results.str.split(' ', expand = True).rename(columns = {0:'goles', 1:'asistencias'}))
resultados_bis = pd.concat([resultados.drop(columns = 'results'), nuevas_columnas], axis = 1)
resultados_bis

,Player,goles,asistencias
0,Lewandowski,12,4
1,Dembélé,3,5
2,Fati,3,3
3,Pedri,2,0


## 4. Datos divididos en varias columnas 

In [15]:
table5 = pd.read_csv("{}table5/table5.csv".format(base_url))
table5

,country,century,year,rate
0,Afghanistan,19,99,745/19987071
1,Afghanistan,20,0,2666/20595360
2,Brazil,19,99,37737/172006362
3,Brazil,20,0,80488/174504898
4,China,19,99,212258/1272915272
5,China,20,0,213766/1280428583


### ¿Cómo lo arreglamos?
![alternative text](imagenes/table5.png)

In [16]:
resultados_bis.assign(result = resultados_bis['goles'] + ' ' +  resultados_bis['asistencias'])
# Alternativamenete: 
#resultados_bis['results'] = resultados_bis['goles'] + ' ' +  resultados_bis['asistencias']
resultados_bis

,Player,goles,asistencias
0,Lewandowski,12,4
1,Dembélé,3,5
2,Fati,3,3
3,Pedri,2,0


### EJERCICIO EN CLASE 
* Arreglar las tablas: table3 y table5
* ¿Qué columnas elegiste?
* ¿Cuántas nuevas columnas necesitaste?
* ¿Qué tokens utilizaste?

## 4. Datos en distintas tablas (joins)

### Llaves:
* **primarias:** identifica de manera única una observación en su propia tabla
* **externas/extranjeras/foreign:** identifica de manera única una observación en otra tabla

#### ¿Cómo encontramos nuestra llave primaria?
* La mayoría de las bases no necesariamente nos van a decir cual o cuales son sus llaves 
* Usaremos las bases de vuelos de la clase de pandas 

In [17]:
planes = pd.read_csv("flights/planes.csv")
#planes.groupby('plane').type.agg(n = 'size').query('n>1')
planes.plane.value_counts().value_counts()

1    2853
Name: plane, dtype: int64

* ¿Qué hicimos acá? 
* ¿Qué necesita ser cierto para que una variable sea la llave primaria?

#### ¿Qué hago si no tengo llave primaria?
* Crear una: puedes usar el número de renglón como llave 

### Relaciones entre tablas: 
* una a una 
* muchas a una
* una a muchas
* muchas a muchas 

In [18]:
goles = pd.DataFrame({'jugador':
  ['Lewandowski', 'Dembélé', 'Fati',
  'Pedri', 'Torres'],
'goles':[12, 3, 3, 2, 2]})
asistencias = pd.DataFrame({'jugador':
  ['Dembélé', 'Lewandowski', 
  'Fati', 'Balde', 'Koundé'],
'asistencias':[5, 4, 3, 3, 2]})

### 1. Inner 
![alternative text](imagenes/inner_join.png)

In [21]:
# merge (Module function)
pd.merge(goles, asistencias, how = 'inner', 
on = 'jugador')
# join (instance method)
#goles.join(asistencias.set_index('jugador'), on = 'jugador', 
#how = 'inner')

,jugador,goles,asistencias
0,Lewandowski,12,4
1,Dembélé,3,5
2,Fati,3,3


#### Diferencias entre `merge` y `join`?
|           | `merge`     | `join`         |
|-----------| ----------- | -------------- |
|junta en:  | columnas    | indices        |
|default:   | inner       | left join      |
|ventajas:  | sin extra   | más eficiente  |


### 2. Outer joins 

Existen 3 tipos de outer joins: 

1. Izquierda
2. Derecha 
3. Full


### 2.1 Outer join: izquierda 
![alternative text](imagenes/left_join.png)


In [22]:
# Merge
# pd.merge(goles, asistencias, how = 'left', on = 'jugador')
# Join
goles.join(asistencias.set_index('jugador'), 
on = 'jugador')

,jugador,goles,asistencias
0,Lewandowski,12,4.0
1,Dembélé,3,5.0
2,Fati,3,3.0
3,Pedri,2,NaN
4,Torres,2,NaN


#### 2.2 Outer join: derecha

![alternative text](imagenes/right_join.png)

In [23]:
pd.merge(goles, asistencias, how = 'right', 
on = 'jugador')

,jugador,goles,asistencias
0,Dembélé,3.0,5
1,Lewandowski,12.0,4
2,Fati,3.0,3
3,Balde,NaN,3
4,Koundé,NaN,2


### 2.3 Outer join: full

![alternative text](imagenes/full_join.png)

In [25]:
pd.merge(goles, asistencias, how = 'outer',
on = 'jugador')

,jugador,goles,asistencias
0,Lewandowski,12.0,4.0
1,Dembélé,3.0,5.0
2,Fati,3.0,3.0
3,Pedri,2.0,NaN
4,Torres,2.0,NaN
5,Balde,NaN,3.0
6,Koundé,NaN,2.0


### 3. Semi joins 
![alternative text](imagenes/semi_join.png)

In [26]:
inner_join = pd.merge(goles, asistencias,on = 'jugador')
goles[goles['jugador'].isin(inner_join['jugador'])]

,jugador,goles
0,Lewandowski,12
1,Dembélé,3
2,Fati,3


### 4. Anti joins 
![alternative text](imagenes/anti_join.png)

In [27]:
outer = pd.merge(goles, asistencias, 
how='outer', indicator=True)
outer[outer._merge == 'left_only'].drop('_merge',axis = 1)

,jugador,goles,asistencias
3,Pedri,2.0,NaN
4,Torres,2.0,NaN


### Tipos de joins: 

* `inner`: incluye unicamente renglones que aparecen en ambas tablas 
* `left`: incluye todos los renglones de `x` y aquellos de `y` que hacen match
* `right`: incluye todos los renglones de `y` y aquellos de `x` que hacen match
* `outer`: incluter todos los renglones de `x` y `y`
* `semi`: incluye los renglones de `x` que hacen match con `y`
* `anti`: incluye los renglones de `x` que no hacen match con `y`

### Sugerencia de como usar joins: 

* **PREFERRED JOINS:** `left` y `inner`
* **NOT THAT COMMON:** `right` y `outer` (con cuidado)
* **CHECAR MESSY JOINS:** `semi` y `anti`

### EJERCICIO EN CLASE:
Utilizando las bases de datos de `flights` contesta lo siguiente:
1. ¿Qué condiciones del clima están asociadas a retrasos de vuelos que salen de Houston?
2. Los aviones más viejos son los que más se retrazan?

# Missing Values 

Cuando volvemos nuestra base tidy pueden aparecer missing values que antes no estabamos viendo y estos pueden ser de dos tipos:

* **Explicitos:** vemos los `NA`
* **Implicitos:** no estan presentes en la base

Usaremos la siguiente base para ilustrar esto: 

In [13]:
data_stocks = {'year':[2015,2015,2015,2015,2016,2016,2016],
              'qtr': [1,2,3,4,2,3,4],
              'retorno':[1.88,0.59,0.35,np.nan,0.92,0.17,2.66]}
stocks = pd.DataFrame(data_stocks)
stocks

,year,qtr,retorno
0,2015,1,1.88
1,2015,2,0.59
2,2015,3,0.35
3,2015,4,NaN
4,2016,2,0.92
5,2016,3,0.17
6,2016,4,2.66


En esta base hay dos missing values: 
   * El return del 4 cuatrimestre del 2015 falta explicitamente, hay un `NaN` ahí
   * El return del primer cuatrimestre del 2016 falta implicitamente, no esta ahí

Depende de como representemos esta base podemos hacer los valores faltantes implicitos, explicitos. 

In [17]:
stocks.pivot_table(index = 'qtr',
                   columns = 'year',
                  values = 'retorno')

year,2015,2016
qtr,,
1,1.88,NaN
2,0.59,0.92
3,0.35,0.17
4,NaN,2.66


Otra manera de encontrar los missing implicitos, es completar la base de datos y sacar la diferencia entre la base completa y la base original. 
1. Completa la base (usando el identificador único de la base)
2. Calcula la diferencia 

In [26]:
## 1. Completar la base (acá el identificar único es el año y el cuatrimestre)
all_years = stocks['year'].unique()
all_qtrs = stocks['qtr'].unique()
complete_df = pd.MultiIndex.from_product([all_years, all_qtrs], names = ['year', 'qtr']).to_frame(index = False)
implicit_stocks = pd.merge(complete_df, stocks, on = ['year', 'qtr'], how = 'left', indicator = True)

## 2. Calcular la diferencia de renglones 
row_difference = len(implicit_stocks) - len(stocks)
print(row_difference)

1


In [27]:
implicit_stocks

,year,qtr,retorno,_merge
0,2015,1,1.88,both
1,2015,2,0.59,both
2,2015,3,0.35,both
3,2015,4,NaN,both
4,2016,1,NaN,left_only
5,2016,2,0.92,both
6,2016,3,0.17,both
7,2016,4,2.66,both
